# Gene → BiologicalProcess Relation Pipeline

Builds a unified, deduplicated edge table for the **Gene–BiologicalProcess** relation.

**Output schema:** `head | relation | tail | head_type | relation_type | tail_type | kg_source | kg_type | head_id_is | tail_id_is | head_detail_name | tail_detail_name`

---
**Sources included:**
- ageanno_processed_mo (AgeAnnoMO_Human_Gene_BiologicalProcess.csv - Native Aging)
- agingatlas (Human/Human_Gene_BiologicalProcess.csv - Native Aging)
- agingbank (4 files - Native Aging)
- cellage (2 files - Native Aging)
- digitalagingatlas (6 files - Native Aging)
- DRKG (DRKG_Gene_Biological_Process.csv - Native Generalised)
- genage (Human/Human_Gene_BiologicalProcess.csv - Native Aging)
- GPKG (Gene_BiologicalProcess_GPKG.csv - Native Generalised)
- Hetionet (Gene_BiologicalProcess_Hetionet.csv - Native Generalised)
- Monarch (Human/Gene_Human_BiologicalProcess_MONARCH.csv - Native Generalised)
- TARKG (Gene_BiologicalProcess_TARKG.csv - Native Generalised)

**Base paths:**
- KG data: `/storage/Arushi/090526_EvoAge/kg_formation/`
- Processed KG files: `/storage/Arushi/090526_EvoAge/kg_formation/processed_data/`

## 0 · Imports & Base Paths

In [1]:
import pandas as pd
import os

BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR     = BASE_DIR + 'processed_data/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/GENE_BIOLOGICALPROCESS/ALL_GENE_BIOLOGICALPROCESS.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type', 'head_id_is', 'tail_id_is', 'head_detail_name', 'tail_detail_name','species'
]

## 1 · Mapping DBs

In [15]:
# MESH mapping to map tail (GO_ID) to name
go_db = pd.read_csv(MAPPING_DIR + 'MESH/MESH_GO_ID_NAME.csv')
GO_dict = dict(zip(go_db['id'], go_db['name']))
del go_db

# NCBI mapping to resolve gene synonyms and descriptions
ncbi = pd.read_csv(MAPPING_DIR + 'ncbi/Homo_sapiens.gene_info', sep='\t')
NCBI_sym2desc_dict = dict(zip(ncbi['Symbol'],  ncbi['description']))
NCBI_syn2sym_dict = {}
for synonyms, symbol in zip(ncbi['Synonyms'], ncbi['Symbol']):
    for syn in str(synonyms).split('|'):
        NCBI_syn2sym_dict[syn.strip()] = symbol
del ncbi
NCBI_sym2desc_dict

{'A1BG': 'alpha-1-B glycoprotein',
 'A2M': 'alpha-2-macroglobulin',
 'A2MP1': 'alpha-2-macroglobulin pseudogene 1',
 'NAT1': 'N-acetyltransferase 1',
 'NAT2': 'N-acetyltransferase 2',
 'NATP': 'N-acetyltransferase pseudogene',
 'SERPINA3': 'serpin family A member 3',
 'AADAC': 'arylacetamide deacetylase',
 'AAMP': 'angio associated migratory cell protein',
 'AANAT': 'aralkylamine N-acetyltransferase',
 'AARS1': 'alanyl-tRNA synthetase 1',
 'AAVS1': 'adeno-associated virus integration site 1',
 'ABAT': '4-aminobutyrate aminotransferase',
 'ABCA1': 'ATP binding cassette subfamily A member 1',
 'ABCA2': 'ATP binding cassette subfamily A member 2',
 'ABCA3': 'ATP binding cassette subfamily A member 3',
 'ABCB7': 'ATP binding cassette subfamily B member 7',
 'ABCF1': 'ATP binding cassette subfamily F member 1',
 'ABCA4': 'ATP binding cassette subfamily A member 4',
 'ABL1': 'ABL proto-oncogene 1, non-receptor tyrosine kinase',
 'AOC1': 'amine oxidase copper containing 1',
 'ABL2': 'ABL prot

## 2 · Load Source Files

# # 1. ageanno_processed_mo (AgeAnnoMO_Human_Gene_BiologicalProcess.csv)


In [3]:
ageanno_path = PROC_DIR + 'ageanno_processed_mo/AgeAnnoMO_Human_Gene_BiologicalProcess.csv'
if os.path.exists(ageanno_path):
    AgeAnno = pd.read_csv(ageanno_path)
    AgeAnno.columns = AgeAnno.columns.str.lower()
    if 'kgsource' in AgeAnno.columns:
        AgeAnno = AgeAnno.rename(columns={'kgsource': 'kg_source'})
    if 'head_name' in AgeAnno.columns:
        AgeAnno = AgeAnno.rename(columns={'head_name': 'head_detail_name'})
    if 'tail_name' in AgeAnno.columns:
        AgeAnno = AgeAnno.rename(columns={'tail_name': 'tail_detail_name'})
    AgeAnno['head_id_is'] = 'NCBI_ID'
    AgeAnno['tail_id_is'] = 'Quick_GO'
    AgeAnno['relation'] = 'Gene_BiologicalProcess'
    AgeAnno['head_type'] = 'Gene'
    AgeAnno['tail_type'] = 'BiologicalProcess'
    AgeAnno['kg_source'] = 'AgeAnnoMO'
    AgeAnno['kg_type'] = 'Aging'
    if 'tail_detail_name' not in AgeAnno.columns:
        AgeAnno['tail_detail_name'] = AgeAnno['tail'].map(GO_dict)
    else:
        AgeAnno['tail_detail_name'] = AgeAnno['tail_detail_name'].fillna(AgeAnno['tail'].map(GO_dict))
    print(f"AgeAnno: {len(AgeAnno):,} rows")
else:
    print("Warning: ageanno_processed_mo path not found")
    AgeAnno = pd.DataFrame(columns=REQUIRED_COLS)
AgeAnno['species'] = 'Homo sapiens'
AgeAnno['kg_type'] = 'Aging'
AgeAnno

AgeAnno: 1,556 rows


,head,relation,tail,head_type,tail_type,relation_source,species,tail_detail_name,head_detail_name,head_id_is,tail_id_is,kg_source,kg_type
0,EIF1,Gene_BiologicalProcess,GO:1111113,Gene,BiologicalProcess,AgeAnnoMO,Homo sapiens,Epigenetic Alterations,eukaryotic translation initiation factor 1,NCBI_ID,Quick_GO,AgeAnnoMO,Aging
1,SELPLG,Gene_BiologicalProcess,GO:1111113,Gene,BiologicalProcess,AgeAnnoMO,Homo sapiens,Epigenetic Alterations,selectin P ligand,NCBI_ID,Quick_GO,AgeAnnoMO,Aging
2,LINC01592,Gene_BiologicalProcess,GO:1111113,Gene,BiologicalProcess,AgeAnnoMO,Homo sapiens,Epigenetic Alterations,long intergenic non-protein coding RNA 1592,NCBI_ID,Quick_GO,AgeAnnoMO,Aging
3,PDE3A,Gene_BiologicalProcess,GO:1111113,Gene,BiologicalProcess,AgeAnnoMO,Homo sapiens,Epigenetic Alterations,phosphodiesterase 3A,NCBI_ID,Quick_GO,AgeAnnoMO,Aging
4,OSBPL3,Gene_BiologicalProcess,GO:1111113,Gene,BiologicalProcess,AgeAnnoMO,Homo sapiens,Epigenetic Alterations,oxysterol binding protein like 3,NCBI_ID,Quick_GO,AgeAnnoMO,Aging
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1551,DUSP5,Gene_BiologicalProcess,GO:1111113,Gene,BiologicalProcess,AgeAnnoMO,Homo sapiens,Epigenetic Alterations,dual specificity phosphatase 5,NCBI_ID,Quick_GO,AgeAnnoMO,Aging
1552,SMIM21,Gene_BiologicalProcess,GO:1111113,Gene,BiologicalProcess,AgeAnnoMO,Homo sapiens,Epigenetic Alterations,small integral membrane protein 21,NCBI_ID,Quick_GO,AgeAnnoMO,Aging
1553,SLC35A1,Gene_BiologicalProcess,GO:1111113,Gene,BiologicalProcess,AgeAnnoMO,Homo sapiens,Epigenetic Alterations,solute carrier family 35 member A1,NCBI_ID,Quick_GO,AgeAnnoMO,Aging
1554,CLGN,Gene_BiologicalProcess,GO:1111113,Gene,BiologicalProcess,AgeAnnoMO,Homo sapiens,Epigenetic Alterations,calmegin,NCBI_ID,Quick_GO,AgeAnnoMO,Aging


# 2. agingatlas (Human/Human_Gene_BiologicalProcess.csv)

In [4]:
agingatlas_path = PROC_DIR + 'agingatlas/Human/Human_Gene_BiologicalProcess.csv'
if os.path.exists(agingatlas_path):
    AgingAtlas = pd.read_csv(agingatlas_path)
    AgingAtlas.columns = AgingAtlas.columns.str.lower()
    if 'kgsource' in AgingAtlas.columns:
        AgingAtlas = AgingAtlas.rename(columns={'kgsource': 'kg_source'})
    if 'head_name' in AgingAtlas.columns:
        AgingAtlas = AgingAtlas.rename(columns={'head_name': 'head_detail_name'})
    if 'tail_name' in AgingAtlas.columns:
        AgingAtlas = AgingAtlas.rename(columns={'tail_name': 'tail_detail_name'})
    if 'relation_source' in AgingAtlas.columns:
        AgingAtlas = AgingAtlas.rename(columns={'relation_source': 'kg_source'})
    AgingAtlas['head_id_is'] = 'NCBI_ID'
    AgingAtlas['tail_id_is'] = 'Quick_GO'
    AgingAtlas['relation'] = 'Gene_BiologicalProcess'
    AgingAtlas['head_type'] = 'Gene'
    AgingAtlas['tail_type'] = 'BiologicalProcess'
    AgingAtlas['kg_source'] = 'AgingAtlas'
    if 'tail_detail_name' not in AgingAtlas.columns:
        AgingAtlas['tail_detail_name'] = AgingAtlas['tail'].map(GO_dict)
    else:
        AgingAtlas['tail_detail_name'] = AgingAtlas['tail_detail_name'].fillna(AgingAtlas['tail'].map(GO_dict))
    print(f"AgingAtlas: {len(AgingAtlas):,} rows")
else:
    print("Warning: agingatlas path not found")
    AgingAtlas = pd.DataFrame(columns=REQUIRED_COLS)
AgingAtlas['species'] = 'Homo sapiens'
AgingAtlas['kg_type'] = 'Aging'
AgingAtlas

AgingAtlas: 503 rows


,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_source,species,symbol,gene_set,kegg_id,kegg_name,literature_name,literature_link,data_type,kg_type
0,IL2RB,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,Interleukin 2 Receptor Subunit Beta,aging,NCBI_ID,Quick_GO,AgingAtlas,Homo sapiens,IL2RB,others,hsa04060,Cytokine-cytokine receptor interaction,Defective CD8 Signaling Pathways Delay Rejecti...,10.1097/TP.0000000000000886,Associative,Aging
1,PSAT1,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,Phosphoserine Aminotransferase 1,aging,NCBI_ID,Quick_GO,AgingAtlas,Homo sapiens,PSAT1,others,hsa00260 hsa00260,"Glycine, serine and threonine metabolism",Serine Metabolism Controls Dental Pulp Stem Ce...,10.1177/0022034520958374,Associative,Aging
2,A2M,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,alpha-2-macroglobulin,aging,NCBI_ID,Quick_GO,AgingAtlas,Homo sapiens,A2M,others,unknown,Unknown,Alpha-2 Macroglobulin Is Genetically Associate...,10.1038/1243,Associative,Aging
3,AARS1,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,Alanyl-TRNA Synthetase 1,aging,NCBI_ID,Quick_GO,AgingAtlas,Homo sapiens,AARS1,loss of proteostasis,hsa00970,Aminoacyl-tRNA biosynthesis,Altered Sensory Neuron Development in CMT2D Mi...,10.3389/fncel.2020.00232,Associative,Aging
4,ABL1,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,"ABL proto-oncogene 1, non-receptor tyrosine ki...",aging,NCBI_ID,Quick_GO,AgingAtlas,Homo sapiens,ABL1,genomic instability,hsa04110,Cell cycle,p73 Is Regulated by Tyrosine Kinase c-Abl in t...,10.1038/21704,Associative,Aging
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,XRCC5,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,X-ray repair complementing defective repair in...,aging,NCBI_ID,Quick_GO,AgingAtlas,Homo sapiens,XRCC5,telomere attrition,unknown,Unknown,Human Ku70/80 Associates Physically With Telom...,10.1074/jbc.M208542200,Associative,Aging
499,XRCC6,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,X-ray repair complementing defective repair in...,aging,NCBI_ID,Quick_GO,AgingAtlas,Homo sapiens,XRCC6,others,unknown,Unknown,Human Ku70/80 associates physically with telom...,10.1074/jbc.M208542200,Associative,Aging
500,YWHAZ,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,tyrosine 3-monooxygenase/tryptophan 5-monooxyg...,aging,NCBI_ID,Quick_GO,AgingAtlas,Homo sapiens,YWHAZ,stem cell exhaustion,hsa04110,Cell cycle,Proteomic Analysis Identifies That 14-3-3zeta ...,10.1073/pnas.0406499101,Associative,Aging
501,ZAP70,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,Zeta Chain Of T Cell Receptor Associated Prote...,aging,NCBI_ID,Quick_GO,AgingAtlas,Homo sapiens,ZAP70,NF-?B related gene,hsa04064,NF-kappa B signaling pathway,Otud7b facilitates T cell activation and infla...,10.1084/jem.20151426,Associative,Aging


### 3 GenAge

In [5]:
genage_path = PROC_DIR + 'genage/Human/Human_Gene_BiologicalProcess.csv'
genage = pd.read_csv(genage_path)
genage['species'] = 'Homo sapiens'
genage['kg_type'] = 'Aging'
genage

,head,relation,tail,head_type,tail_type,kg_source,head_id_is,tail_id_is,relation_type,head_detail_name,tail_detail_name,species,kg_type
0,GHR,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,GenAge,NCBI_ID,Quick_GO,Causative,growth hormone receptor,aging,Homo sapiens,Aging
1,GHRH,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,GenAge,NCBI_ID,Quick_GO,Causative,growth hormone releasing hormone,aging,Homo sapiens,Aging
2,SHC1,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,GenAge,NCBI_ID,Quick_GO,Causative,SHC adaptor protein 1,aging,Homo sapiens,Aging
3,POU1F1,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,GenAge,NCBI_ID,Quick_GO,Causative,POU class 1 homeobox 1,aging,Homo sapiens,Aging
4,PROP1,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,GenAge,NCBI_ID,Quick_GO,Causative,PROP paired-like homeobox 1,aging,Homo sapiens,Aging
...,...,...,...,...,...,...,...,...,...,...,...,...,...
297,TRAP1,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,GenAge,NCBI_ID,Quick_GO,Causative,TNF receptor associated protein 1,aging,Homo sapiens,Aging
298,TRPV1,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,GenAge,NCBI_ID,Quick_GO,Causative,transient receptor potential cation channel su...,aging,Homo sapiens,Aging
299,NFE2L1,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,GenAge,NCBI_ID,Quick_GO,Causative,NFE2 like bZIP transcription factor 1,aging,Homo sapiens,Aging
300,IFNB1,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,GenAge,NCBI_ID,Quick_GO,Causative,interferon beta 1,aging,Homo sapiens,Aging


## 4 DRKG

In [6]:
drkg_path = PROC_DIR + 'DRKG/DRKG_Gene_Biological_Process.csv'
if os.path.exists(drkg_path):
    DRKG = pd.read_csv(drkg_path)
    DRKG.columns = DRKG.columns.str.lower()
    if 'head_name' in DRKG.columns:
        DRKG = DRKG.rename(columns={'head_name': 'head_detail_name'})
    if 'tail_name' in DRKG.columns:
        DRKG = DRKG.rename(columns={'tail_name': 'tail_detail_name'})
    DRKG['head_id_is'] = 'NCBI_ID'
    DRKG['tail_id_is'] = 'Quick_GO'
    DRKG['relation'] = 'Gene_BiologicalProcess'
    DRKG['head_type'] = 'Gene'
    DRKG['tail_type'] = 'BiologicalProcess'
    DRKG['kg_source'] = 'DRKG'
    DRKG['kg_type'] = 'Generalised'
    DRKG['tail_detail_name'] = DRKG['tail'].map(GO_dict)
    print(f"DRKG: {len(DRKG):,} rows")
else:
    print("Warning: DRKG path not found")
    DRKG = pd.DataFrame(columns=REQUIRED_COLS)
DRKG['species'] = 'Homo sapiens'
DRKG['kg_type'] = 'Generalised'
DRKG

DRKG: 559,453 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,head_detail_name,head_ens,tail_detail_name,head_id_is,tail_id_is,head_orignal,tail_orignal,kg_type,species
0,SOCS3,Gene_BiologicalProcess,GO:0071357,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,DRKG,9021,suppressor of cytokine signaling 3,ENSG00000184557,cellular response to type I interferon,NCBI_ID,Quick_GO,Gene::9021,Biological Process::GO:0071357,Generalised,Homo sapiens
1,ASB2,Gene_BiologicalProcess,GO:0098780,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,DRKG,51676,ankyrin repeat and SOCS box containing 2,ENSG00000100628,response to mitochondrial depolarisation,NCBI_ID,Quick_GO,Gene::51676,Biological Process::GO:0098780,Generalised,Homo sapiens
2,ABCA1,Gene_BiologicalProcess,GO:0055088,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,DRKG,19,ATP binding cassette subfamily A member 1,ENSG00000165029,lipid homeostasis,NCBI_ID,Quick_GO,Gene::19,Biological Process::GO:0055088,Generalised,Homo sapiens
3,HNMT,Gene_BiologicalProcess,GO:0010243,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,DRKG,3176,histamine N-methyltransferase,ENSG00000150540,obsolete response to organonitrogen compound,NCBI_ID,Quick_GO,Gene::3176,Biological Process::GO:0010243,Generalised,Homo sapiens
4,HBA1,Gene_BiologicalProcess,GO:0006898,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,DRKG,3039,hemoglobin subunit alpha 1,ENSG00000206172,receptor-mediated endocytosis,NCBI_ID,Quick_GO,Gene::3039,Biological Process::GO:0006898,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
559448,SLC35B2,Gene_BiologicalProcess,GO:0042451,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,DRKG,347734,solute carrier family 35 member B2,ENSG00000157593,purine nucleoside biosynthetic process,NCBI_ID,Quick_GO,Gene::347734,Biological Process::GO:0042451,Generalised,Homo sapiens
559449,SCFD2,Gene_BiologicalProcess,GO:0048278,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,DRKG,152579,sec1 family domain containing 2,ENSG00000184178,vesicle docking,NCBI_ID,Quick_GO,Gene::152579,Biological Process::GO:0048278,Generalised,Homo sapiens
559450,APOM,Gene_BiologicalProcess,GO:0097006,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,DRKG,55937,apolipoprotein M,ENSG00000204444,regulation of plasma lipoprotein particle levels,NCBI_ID,Quick_GO,Gene::55937,Biological Process::GO:0097006,Generalised,Homo sapiens
559451,CYP2R1,Gene_BiologicalProcess,GO:0044107,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,DRKG,120227,cytochrome P450 family 2 subfamily R member 1,ENSG00000186104,obsolete cellular alcohol metabolic process,NCBI_ID,Quick_GO,Gene::120227,Biological Process::GO:0044107,Generalised,Homo sapiens



# 5. GPKG

In [7]:

gpkg_path = PROC_DIR + 'GPKG/Gene_BiologicalProcess_GPKG.csv'
if os.path.exists(gpkg_path):
    GPKG = pd.read_csv(gpkg_path)
    GPKG.columns = GPKG.columns.str.lower()
    if 'head_name' in GPKG.columns:
        GPKG = GPKG.rename(columns={'head_name': 'head_detail_name'})
    if 'tail_name' in GPKG.columns:
        GPKG = GPKG.rename(columns={'tail_name': 'tail_detail_name'})
    GPKG['head_id_is'] = 'NCBI_ID'
    GPKG['tail_id_is'] = 'Quick_GO'
    GPKG['relation'] = 'Gene_BiologicalProcess'
    GPKG['head_type'] = 'Gene'
    GPKG['tail_type'] = 'BiologicalProcess'
    GPKG['kg_source'] = 'GPKG'
    GPKG['kg_type'] = 'Generalised'
    if 'tail_detail_name' not in GPKG.columns:
        GPKG['tail_detail_name'] = GPKG['tail'].map(GO_dict)
    else:
        GPKG['tail_detail_name'] = GPKG['tail_detail_name'].fillna(GPKG['tail'].map(GO_dict))
    print(f"GPKG: {len(GPKG):,} rows")
else:
    print("Warning: GPKG path not found")
    GPKG = pd.DataFrame(columns=REQUIRED_COLS)
GPKG['species'] = 'Homo sapiens'
GPKG['kg_type'] = 'Generalised'
GPKG

GPKG: 93,395 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_orignal,head_alias_mapped,head_detail_name,head_id_is,tail_detail_name,tail_id_is,kg_type,species
0,TRAV27,Gene_BiologicalProcess,GO:0050830,Gene,associated,BiologicalProcess,GPKG,ENSG00000211809,TRAV27,T cell receptor alpha variable 27,NCBI_ID,defense response to Gram-positive bacterium,Quick_GO,Generalised,Homo sapiens
1,TRAV27,Gene_BiologicalProcess,GO:0009617,Gene,associated,BiologicalProcess,GPKG,ENSG00000211809,TRAV27,T cell receptor alpha variable 27,NCBI_ID,response to bacterium,Quick_GO,Generalised,Homo sapiens
2,TRAV27,Gene_BiologicalProcess,GO:0006955,Gene,associated,BiologicalProcess,GPKG,ENSG00000211809,TRAV27,T cell receptor alpha variable 27,NCBI_ID,immune response,Quick_GO,Generalised,Homo sapiens
3,MEIKIN,Gene_BiologicalProcess,GO:0051754,Gene,associated,BiologicalProcess,GPKG,ENSG00000239642,MEIKIN,meiotic kinetochore factor,NCBI_ID,"meiotic sister chromatid cohesion, centromeric",Quick_GO,Generalised,Homo sapiens
4,MEIKIN,Gene_BiologicalProcess,GO:0007060,Gene,associated,BiologicalProcess,GPKG,ENSG00000239642,MEIKIN,meiotic kinetochore factor,NCBI_ID,male meiosis chromosome segregation,Quick_GO,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93390,CAPN12,Gene_BiologicalProcess,GO:0006508,Gene,associated,BiologicalProcess,GPKG,ENSG00000182472,CAPN12,calpain 12,NCBI_ID,proteolysis,Quick_GO,Generalised,Homo sapiens
93391,RNF121,Gene_BiologicalProcess,GO:0030433,Gene,associated,BiologicalProcess,GPKG,ENSG00000137522,RNF121,ring finger protein 121,NCBI_ID,obsolete ubiquitin-dependent ERAD pathway,Quick_GO,Generalised,Homo sapiens
93392,DHDH,Gene_BiologicalProcess,GO:0042843,Gene,associated,BiologicalProcess,GPKG,ENSG00000104808,DHDH,dihydrodiol dehydrogenase,NCBI_ID,D-xylose catabolic process,Quick_GO,Generalised,Homo sapiens
93393,GPRIN2,Gene_BiologicalProcess,GO:0031175,Gene,associated,BiologicalProcess,GPKG,ENSG00000204175,GPRIN2,G protein regulated inducer of neurite outgrow...,NCBI_ID,neuron projection development,Quick_GO,Generalised,Homo sapiens


## 6. Hetionet

In [8]:
hetionet_path = PROC_DIR + 'Hetionet/Gene_BiologicalProcess_Hetionet.csv'
if os.path.exists(hetionet_path):
    Hetionet = pd.read_csv(hetionet_path)
    Hetionet.columns = Hetionet.columns.str.lower()
    if 'tail_name' in Hetionet.columns:
        Hetionet = Hetionet.rename(columns={'tail_name': 'tail_detail_name'})
    Hetionet['head_id_is'] = 'NCBI_ID'
    Hetionet['tail_id_is'] = 'Quick_GO'
    Hetionet['relation'] = 'Gene_BiologicalProcess'
    Hetionet['head_type'] = 'Gene'
    Hetionet['tail_type'] = 'BiologicalProcess'
    Hetionet['kg_source'] = 'Hetionet'
    Hetionet['kg_type'] = 'Generalised'
    if 'tail_detail_name' not in Hetionet.columns:
        Hetionet['tail_detail_name'] = Hetionet['tail'].map(GO_dict)
    else:
        Hetionet['tail_detail_name'] = Hetionet['tail_detail_name'].fillna(Hetionet['tail'].map(GO_dict))
    print(f"Hetionet: {len(Hetionet):,} rows")
else:
    print("Warning: Hetionet path not found")
    Hetionet = pd.DataFrame(columns=REQUIRED_COLS)

Hetionet['species'] = 'Homo sapiens'
Hetionet['kg_type'] = 'Generalised'
Hetionet

Hetionet: 559,453 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_name,head_detail_name,tail_detail_name,kg_type,species
0,SOCS3,Gene_BiologicalProcess,GO:0071357,Gene,Gene–participates–BiologicalProcess,BiologicalProcess,Hetionet,NCBI_ID,Quick_GO,9021,suppressor of cytokine signaling 3,cellular response to type I interferon,Generalised,Homo sapiens
1,ASB2,Gene_BiologicalProcess,GO:0098780,Gene,Gene–participates–BiologicalProcess,BiologicalProcess,Hetionet,NCBI_ID,Quick_GO,51676,ankyrin repeat and SOCS box containing 2,response to mitochondrial depolarisation,Generalised,Homo sapiens
2,ABCA1,Gene_BiologicalProcess,GO:0055088,Gene,Gene–participates–BiologicalProcess,BiologicalProcess,Hetionet,NCBI_ID,Quick_GO,19,ATP binding cassette subfamily A member 1,lipid homeostasis,Generalised,Homo sapiens
3,HNMT,Gene_BiologicalProcess,GO:0010243,Gene,Gene–participates–BiologicalProcess,BiologicalProcess,Hetionet,NCBI_ID,Quick_GO,3176,histamine N-methyltransferase,obsolete response to organonitrogen compound,Generalised,Homo sapiens
4,HBA1,Gene_BiologicalProcess,GO:0006898,Gene,Gene–participates–BiologicalProcess,BiologicalProcess,Hetionet,NCBI_ID,Quick_GO,3039,hemoglobin subunit alpha 1,receptor-mediated endocytosis,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
559448,SLC35B2,Gene_BiologicalProcess,GO:0042451,Gene,Gene–participates–BiologicalProcess,BiologicalProcess,Hetionet,NCBI_ID,Quick_GO,347734,solute carrier family 35 member B2,purine nucleoside biosynthetic process,Generalised,Homo sapiens
559449,SCFD2,Gene_BiologicalProcess,GO:0048278,Gene,Gene–participates–BiologicalProcess,BiologicalProcess,Hetionet,NCBI_ID,Quick_GO,152579,sec1 family domain containing 2,vesicle docking,Generalised,Homo sapiens
559450,APOM,Gene_BiologicalProcess,GO:0097006,Gene,Gene–participates–BiologicalProcess,BiologicalProcess,Hetionet,NCBI_ID,Quick_GO,55937,apolipoprotein M,regulation of plasma lipoprotein particle levels,Generalised,Homo sapiens
559451,CYP2R1,Gene_BiologicalProcess,GO:0044107,Gene,Gene–participates–BiologicalProcess,BiologicalProcess,Hetionet,NCBI_ID,Quick_GO,120227,cytochrome P450 family 2 subfamily R member 1,obsolete cellular alcohol metabolic process,Generalised,Homo sapiens


# 7. Monarch

In [9]:
monarch_path = PROC_DIR + 'Monarch/Human/Gene_Human_BiologicalProcess_MONARCH.csv'
if os.path.exists(monarch_path):
    Monarch = pd.read_csv(monarch_path)
    Monarch.columns = Monarch.columns.str.lower()
    if 'kgsource' in Monarch.columns:
        Monarch = Monarch.rename(columns={'kgsource': 'kg_source'})
    if 'head_name' in Monarch.columns:
        Monarch = Monarch.rename(columns={'head_name': 'head_detail_name'})
    if 'tail_name' in Monarch.columns:
        Monarch = Monarch.rename(columns={'tail_name': 'tail_detail_name'})
    Monarch['head_id_is'] = 'NCBI_ID'
    Monarch['tail_id_is'] = 'Quick_GO'
    Monarch['relation'] = 'Gene_BiologicalProcess'
    Monarch['head_type'] = 'Gene'
    Monarch['tail_type'] = 'BiologicalProcess'
    Monarch['kg_source'] = 'Monarch_KG'
    Monarch['kg_type'] = 'Generalised'
    if 'tail_detail_name' not in Monarch.columns:
        Monarch['tail_detail_name'] = Monarch['tail'].map(GO_dict)
    else:
        Monarch['tail_detail_name'] = Monarch['tail_detail_name'].fillna(Monarch['tail'].map(GO_dict))
    print(f"Monarch: {len(Monarch):,} rows")
else:
    print("Warning: Monarch path not found")
    Monarch = pd.DataFrame(columns=REQUIRED_COLS)
Monarch['species'] = 'Homo sapiens'
Monarch['kg_type'] = 'Generalised'
Monarch

Monarch: 166,291 rows


,head,relation,tail,head_type,relation_type,tail_type,relation_source,kg_source,head_detail_name,head_taxon_name,tail_taxon_name,head_id_is,tail_id_is,head_detail_name,tail_detail_name,head_orignal,species_species,kg_type,species
0,ADRA1B,Gene_BiologicalProcess,GO:0007200,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Monarch_KG,adrenoceptor alpha 1B,Homo sapiens,NaN,NCBI_ID,Quick_GO,ADRA1B,phospholipase C-activating G protein-coupled r...,HGNC:278,Homo sapiens_Homo sapiens,Generalised,Homo sapiens
1,ATP13A4,Gene_BiologicalProcess,GO:0006874,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Monarch_KG,ATPase 13A4,Homo sapiens,NaN,NCBI_ID,Quick_GO,ATP13A4,intracellular calcium ion homeostasis,HGNC:25422,Homo sapiens_Homo sapiens,Generalised,Homo sapiens
2,TGIF2,Gene_BiologicalProcess,GO:0000122,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Monarch_KG,TGFB induced factor homeobox 2,Homo sapiens,NaN,NCBI_ID,Quick_GO,TGIF2,negative regulation of transcription by RNA po...,HGNC:15764,Homo sapiens_Homo sapiens,Generalised,Homo sapiens
3,CACNG7,Gene_BiologicalProcess,GO:0051968,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Monarch_KG,calcium voltage-gated channel auxiliary subuni...,Homo sapiens,NaN,NCBI_ID,Quick_GO,CACNG7,"positive regulation of synaptic transmission, ...",HGNC:13626,Homo sapiens_Homo sapiens,Generalised,Homo sapiens
4,IGKV2D-28,Gene_BiologicalProcess,GO:0006955,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Monarch_KG,immunoglobulin kappa variable 2D-28,Homo sapiens,NaN,NCBI_ID,Quick_GO,IGKV2D-28,immune response,HGNC:5799,Homo sapiens_Homo sapiens,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166286,HIKESHI,Gene_BiologicalProcess,GO:0006606,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Monarch_KG,heat shock protein nuclear import factor hikeshi,Homo sapiens,NaN,NCBI_ID,Quick_GO,HIKESHI,protein import into nucleus,HGNC:26938,Homo sapiens_Homo sapiens,Generalised,Homo sapiens
166287,TOP3A,Gene_BiologicalProcess,GO:0006310,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Monarch_KG,DNA topoisomerase III alpha,Homo sapiens,NaN,NCBI_ID,Quick_GO,TOP3A,DNA recombination,HGNC:11992,Homo sapiens_Homo sapiens,Generalised,Homo sapiens
166288,CCDC93,Gene_BiologicalProcess,GO:0006893,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Monarch_KG,coiled-coil domain containing 93,Homo sapiens,NaN,NCBI_ID,Quick_GO,CCDC93,Golgi to plasma membrane transport,HGNC:25611,Homo sapiens_Homo sapiens,Generalised,Homo sapiens
166289,SPTSSB,Gene_BiologicalProcess,GO:0007029,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Monarch_KG,serine palmitoyltransferase small subunit B,Homo sapiens,NaN,NCBI_ID,Quick_GO,SPTSSB,endoplasmic reticulum organization,HGNC:24045,Homo sapiens_Homo sapiens,Generalised,Homo sapiens


## 8. TARKG

In [10]:
tarkg_path = PROC_DIR + 'TARKG/Gene_BiologicalProcess_TARKG.csv'
if os.path.exists(tarkg_path):
    TARKG = pd.read_csv(tarkg_path)
    TARKG.columns = TARKG.columns.str.lower()
    if 'head_name' in TARKG.columns:
        TARKG = TARKG.rename(columns={'head_name': 'head_detail_name'})
    if 'tail_name' in TARKG.columns:
        TARKG = TARKG.rename(columns={'tail_name': 'tail_detail_name'})
    TARKG['head_id_is'] = 'NCBI_ID'
    TARKG['tail_id_is'] = 'Quick_GO'
    TARKG['relation'] = 'Gene_BiologicalProcess'
    TARKG['head_type'] = 'Gene'
    TARKG['tail_type'] = 'BiologicalProcess'
    TARKG['kg_source'] = 'TARKG'
    TARKG['kg_type'] = 'Generalised'
    if 'tail_detail_name' not in TARKG.columns:
        TARKG['tail_detail_name'] = TARKG['tail'].map(GO_dict)
    else:
        TARKG['tail_detail_name'] = TARKG['tail_detail_name'].fillna(TARKG['tail'].map(GO_dict))
    print(f"TARKG: {len(TARKG):,} rows")
else:
    print("Warning: TARKG path not found")
    TARKG = pd.DataFrame(columns=REQUIRED_COLS)
TARKG['species'] = 'Homo sapiens'
TARKG

TARKG: 1,045,588 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species
0,AHCTF1,Gene_BiologicalProcess,GO:0071705,Gene,GpBP,BiologicalProcess,TARKG,AT-hook containing transcription factor 1,nitrogen compound transport,NCBI_ID,Quick_GO,Hetionet-183676,Hetionet,0,Generalised,Homo sapiens
1,AHCTF1,Gene_BiologicalProcess,GO:0071705,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,TARKG,AT-hook containing transcription factor 1,nitrogen compound transport,NCBI_ID,Quick_GO,DRKG-2054878,DRKG,0,Generalised,Homo sapiens
2,NUP62,Gene_BiologicalProcess,GO:0071705,Gene,GpBP,BiologicalProcess,TARKG,nucleoporin 62,nitrogen compound transport,NCBI_ID,Quick_GO,Hetionet-532807,Hetionet,0,Generalised,Homo sapiens
3,NUP62,Gene_BiologicalProcess,GO:0071705,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,TARKG,nucleoporin 62,nitrogen compound transport,NCBI_ID,Quick_GO,DRKG-2404009,DRKG,0,Generalised,Homo sapiens
4,NUP37,Gene_BiologicalProcess,GO:0071705,Gene,GpBP,BiologicalProcess,TARKG,nucleoporin 37,nitrogen compound transport,NCBI_ID,Quick_GO,Hetionet-149653,Hetionet,0,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1045583,SLC35B4,Gene_BiologicalProcess,GO:1990569,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,TARKG,solute carrier family 35 member B4,UDP-N-acetylglucosamine transmembrane transport,NCBI_ID,Quick_GO,DRKG-2332823,DRKG,0,Generalised,Homo sapiens
1045584,PCYOX1,Gene_BiologicalProcess,GO:0030328,Gene,GpBP,BiologicalProcess,TARKG,prenylcysteine oxidase 1,prenylcysteine catabolic process,NCBI_ID,Quick_GO,Hetionet-295606,Hetionet,0,Generalised,Homo sapiens
1045585,PCYOX1,Gene_BiologicalProcess,GO:0030328,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,TARKG,prenylcysteine oxidase 1,prenylcysteine catabolic process,NCBI_ID,Quick_GO,DRKG-2166808,DRKG,0,Generalised,Homo sapiens
1045586,PCYOX1L,Gene_BiologicalProcess,GO:0030328,Gene,GpBP,BiologicalProcess,TARKG,prenylcysteine oxidase 1 like,prenylcysteine catabolic process,NCBI_ID,Quick_GO,Hetionet-177161,Hetionet,0,Generalised,Homo sapiens


In [11]:
source_dfs = {
    "AgeAnno": AgeAnno,
    "AgingAtlas": AgingAtlas,
    "DRKG": DRKG,
    "genage": genage,
    "GPKG": GPKG,
    "Hetionet": Hetionet,
    "Monarch": Monarch,
    "TARKG": TARKG
}

for name, df in source_dfs.items():
    print(f"name: {name}")
    display(df)   # or just print(df)


name: AgeAnno


,head,relation,tail,head_type,tail_type,relation_source,species,tail_detail_name,head_detail_name,head_id_is,tail_id_is,kg_source,kg_type
0,EIF1,Gene_BiologicalProcess,GO:1111113,Gene,BiologicalProcess,AgeAnnoMO,Homo sapiens,Epigenetic Alterations,eukaryotic translation initiation factor 1,NCBI_ID,Quick_GO,AgeAnnoMO,Aging
1,SELPLG,Gene_BiologicalProcess,GO:1111113,Gene,BiologicalProcess,AgeAnnoMO,Homo sapiens,Epigenetic Alterations,selectin P ligand,NCBI_ID,Quick_GO,AgeAnnoMO,Aging
2,LINC01592,Gene_BiologicalProcess,GO:1111113,Gene,BiologicalProcess,AgeAnnoMO,Homo sapiens,Epigenetic Alterations,long intergenic non-protein coding RNA 1592,NCBI_ID,Quick_GO,AgeAnnoMO,Aging
3,PDE3A,Gene_BiologicalProcess,GO:1111113,Gene,BiologicalProcess,AgeAnnoMO,Homo sapiens,Epigenetic Alterations,phosphodiesterase 3A,NCBI_ID,Quick_GO,AgeAnnoMO,Aging
4,OSBPL3,Gene_BiologicalProcess,GO:1111113,Gene,BiologicalProcess,AgeAnnoMO,Homo sapiens,Epigenetic Alterations,oxysterol binding protein like 3,NCBI_ID,Quick_GO,AgeAnnoMO,Aging
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1551,DUSP5,Gene_BiologicalProcess,GO:1111113,Gene,BiologicalProcess,AgeAnnoMO,Homo sapiens,Epigenetic Alterations,dual specificity phosphatase 5,NCBI_ID,Quick_GO,AgeAnnoMO,Aging
1552,SMIM21,Gene_BiologicalProcess,GO:1111113,Gene,BiologicalProcess,AgeAnnoMO,Homo sapiens,Epigenetic Alterations,small integral membrane protein 21,NCBI_ID,Quick_GO,AgeAnnoMO,Aging
1553,SLC35A1,Gene_BiologicalProcess,GO:1111113,Gene,BiologicalProcess,AgeAnnoMO,Homo sapiens,Epigenetic Alterations,solute carrier family 35 member A1,NCBI_ID,Quick_GO,AgeAnnoMO,Aging
1554,CLGN,Gene_BiologicalProcess,GO:1111113,Gene,BiologicalProcess,AgeAnnoMO,Homo sapiens,Epigenetic Alterations,calmegin,NCBI_ID,Quick_GO,AgeAnnoMO,Aging


name: AgingAtlas


,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_source,species,symbol,gene_set,kegg_id,kegg_name,literature_name,literature_link,data_type,kg_type
0,IL2RB,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,Interleukin 2 Receptor Subunit Beta,aging,NCBI_ID,Quick_GO,AgingAtlas,Homo sapiens,IL2RB,others,hsa04060,Cytokine-cytokine receptor interaction,Defective CD8 Signaling Pathways Delay Rejecti...,10.1097/TP.0000000000000886,Associative,Aging
1,PSAT1,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,Phosphoserine Aminotransferase 1,aging,NCBI_ID,Quick_GO,AgingAtlas,Homo sapiens,PSAT1,others,hsa00260 hsa00260,"Glycine, serine and threonine metabolism",Serine Metabolism Controls Dental Pulp Stem Ce...,10.1177/0022034520958374,Associative,Aging
2,A2M,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,alpha-2-macroglobulin,aging,NCBI_ID,Quick_GO,AgingAtlas,Homo sapiens,A2M,others,unknown,Unknown,Alpha-2 Macroglobulin Is Genetically Associate...,10.1038/1243,Associative,Aging
3,AARS1,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,Alanyl-TRNA Synthetase 1,aging,NCBI_ID,Quick_GO,AgingAtlas,Homo sapiens,AARS1,loss of proteostasis,hsa00970,Aminoacyl-tRNA biosynthesis,Altered Sensory Neuron Development in CMT2D Mi...,10.3389/fncel.2020.00232,Associative,Aging
4,ABL1,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,"ABL proto-oncogene 1, non-receptor tyrosine ki...",aging,NCBI_ID,Quick_GO,AgingAtlas,Homo sapiens,ABL1,genomic instability,hsa04110,Cell cycle,p73 Is Regulated by Tyrosine Kinase c-Abl in t...,10.1038/21704,Associative,Aging
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,XRCC5,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,X-ray repair complementing defective repair in...,aging,NCBI_ID,Quick_GO,AgingAtlas,Homo sapiens,XRCC5,telomere attrition,unknown,Unknown,Human Ku70/80 Associates Physically With Telom...,10.1074/jbc.M208542200,Associative,Aging
499,XRCC6,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,X-ray repair complementing defective repair in...,aging,NCBI_ID,Quick_GO,AgingAtlas,Homo sapiens,XRCC6,others,unknown,Unknown,Human Ku70/80 associates physically with telom...,10.1074/jbc.M208542200,Associative,Aging
500,YWHAZ,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,tyrosine 3-monooxygenase/tryptophan 5-monooxyg...,aging,NCBI_ID,Quick_GO,AgingAtlas,Homo sapiens,YWHAZ,stem cell exhaustion,hsa04110,Cell cycle,Proteomic Analysis Identifies That 14-3-3zeta ...,10.1073/pnas.0406499101,Associative,Aging
501,ZAP70,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,Zeta Chain Of T Cell Receptor Associated Prote...,aging,NCBI_ID,Quick_GO,AgingAtlas,Homo sapiens,ZAP70,NF-?B related gene,hsa04064,NF-kappa B signaling pathway,Otud7b facilitates T cell activation and infla...,10.1084/jem.20151426,Associative,Aging


name: DRKG


...

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_name,head_detail_name,tail_detail_name,kg_type,species
0,SOCS3,Gene_BiologicalProcess,GO:0071357,Gene,Gene–participates–BiologicalProcess,BiologicalProcess,Hetionet,NCBI_ID,Quick_GO,9021,suppressor of cytokine signaling 3,cellular response to type I interferon,Generalised,Homo sapiens
1,ASB2,Gene_BiologicalProcess,GO:0098780,Gene,Gene–participates–BiologicalProcess,BiologicalProcess,Hetionet,NCBI_ID,Quick_GO,51676,ankyrin repeat and SOCS box containing 2,response to mitochondrial depolarisation,Generalised,Homo sapiens
2,ABCA1,Gene_BiologicalProcess,GO:0055088,Gene,Gene–participates–BiologicalProcess,BiologicalProcess,Hetionet,NCBI_ID,Quick_GO,19,ATP binding cassette subfamily A member 1,lipid homeostasis,Generalised,Homo sapiens
3,HNMT,Gene_BiologicalProcess,GO:0010243,Gene,Gene–participates–BiologicalProcess,BiologicalProcess,Hetionet,NCBI_ID,Quick_GO,3176,histamine N-methyltransferase,obsolete response to organonitrogen compound,Generalised,Homo sapiens
4,HBA1,Gene_BiologicalProcess,GO:0006898,Gene,Gene–participates–BiologicalProcess,BiologicalProcess,Hetionet,NCBI_ID,Quick_GO,3039,hemoglobin subunit alpha 1,receptor-mediated endocytosis,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
559448,SLC35B2,Gene_BiologicalProcess,GO:0042451,Gene,Gene–participates–BiologicalProcess,BiologicalProcess,Hetionet,NCBI_ID,Quick_GO,347734,solute carrier family 35 member B2,purine nucleoside biosynthetic process,Generalised,Homo sapiens
559449,SCFD2,Gene_BiologicalProcess,GO:0048278,Gene,Gene–participates–BiologicalProcess,BiologicalProcess,Hetionet,NCBI_ID,Quick_GO,152579,sec1 family domain containing 2,vesicle docking,Generalised,Homo sapiens
559450,APOM,Gene_BiologicalProcess,GO:0097006,Gene,Gene–participates–BiologicalProcess,BiologicalProcess,Hetionet,NCBI_ID,Quick_GO,55937,apolipoprotein M,regulation of plasma lipoprotein particle levels,Generalised,Homo sapiens
559451,CYP2R1,Gene_BiologicalProcess,GO:0044107,Gene,Gene–participates–BiologicalProcess,BiologicalProcess,Hetionet,NCBI_ID,Quick_GO,120227,cytochrome P450 family 2 subfamily R member 1,obsolete cellular alcohol metabolic process,Generalised,Homo sapiens


name: Monarch


,head,relation,tail,head_type,relation_type,tail_type,relation_source,kg_source,head_detail_name,head_taxon_name,tail_taxon_name,head_id_is,tail_id_is,head_detail_name,tail_detail_name,head_orignal,species_species,kg_type,species
0,ADRA1B,Gene_BiologicalProcess,GO:0007200,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Monarch_KG,adrenoceptor alpha 1B,Homo sapiens,NaN,NCBI_ID,Quick_GO,ADRA1B,phospholipase C-activating G protein-coupled r...,HGNC:278,Homo sapiens_Homo sapiens,Generalised,Homo sapiens
1,ATP13A4,Gene_BiologicalProcess,GO:0006874,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Monarch_KG,ATPase 13A4,Homo sapiens,NaN,NCBI_ID,Quick_GO,ATP13A4,intracellular calcium ion homeostasis,HGNC:25422,Homo sapiens_Homo sapiens,Generalised,Homo sapiens
2,TGIF2,Gene_BiologicalProcess,GO:0000122,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Monarch_KG,TGFB induced factor homeobox 2,Homo sapiens,NaN,NCBI_ID,Quick_GO,TGIF2,negative regulation of transcription by RNA po...,HGNC:15764,Homo sapiens_Homo sapiens,Generalised,Homo sapiens
3,CACNG7,Gene_BiologicalProcess,GO:0051968,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Monarch_KG,calcium voltage-gated channel auxiliary subuni...,Homo sapiens,NaN,NCBI_ID,Quick_GO,CACNG7,"positive regulation of synaptic transmission, ...",HGNC:13626,Homo sapiens_Homo sapiens,Generalised,Homo sapiens
4,IGKV2D-28,Gene_BiologicalProcess,GO:0006955,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Monarch_KG,immunoglobulin kappa variable 2D-28,Homo sapiens,NaN,NCBI_ID,Quick_GO,IGKV2D-28,immune response,HGNC:5799,Homo sapiens_Homo sapiens,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166286,HIKESHI,Gene_BiologicalProcess,GO:0006606,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Monarch_KG,heat shock protein nuclear import factor hikeshi,Homo sapiens,NaN,NCBI_ID,Quick_GO,HIKESHI,protein import into nucleus,HGNC:26938,Homo sapiens_Homo sapiens,Generalised,Homo sapiens
166287,TOP3A,Gene_BiologicalProcess,GO:0006310,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Monarch_KG,DNA topoisomerase III alpha,Homo sapiens,NaN,NCBI_ID,Quick_GO,TOP3A,DNA recombination,HGNC:11992,Homo sapiens_Homo sapiens,Generalised,Homo sapiens
166288,CCDC93,Gene_BiologicalProcess,GO:0006893,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Monarch_KG,coiled-coil domain containing 93,Homo sapiens,NaN,NCBI_ID,Quick_GO,CCDC93,Golgi to plasma membrane transport,HGNC:25611,Homo sapiens_Homo sapiens,Generalised,Homo sapiens
166289,SPTSSB,Gene_BiologicalProcess,GO:0007029,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Monarch_KG,serine palmitoyltransferase small subunit B,Homo sapiens,NaN,NCBI_ID,Quick_GO,SPTSSB,endoplasmic reticulum organization,HGNC:24045,Homo sapiens_Homo sapiens,Generalised,Homo sapiens


name: TARKG


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species
0,AHCTF1,Gene_BiologicalProcess,GO:0071705,Gene,GpBP,BiologicalProcess,TARKG,AT-hook containing transcription factor 1,nitrogen compound transport,NCBI_ID,Quick_GO,Hetionet-183676,Hetionet,0,Generalised,Homo sapiens
1,AHCTF1,Gene_BiologicalProcess,GO:0071705,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,TARKG,AT-hook containing transcription factor 1,nitrogen compound transport,NCBI_ID,Quick_GO,DRKG-2054878,DRKG,0,Generalised,Homo sapiens
2,NUP62,Gene_BiologicalProcess,GO:0071705,Gene,GpBP,BiologicalProcess,TARKG,nucleoporin 62,nitrogen compound transport,NCBI_ID,Quick_GO,Hetionet-532807,Hetionet,0,Generalised,Homo sapiens
3,NUP62,Gene_BiologicalProcess,GO:0071705,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,TARKG,nucleoporin 62,nitrogen compound transport,NCBI_ID,Quick_GO,DRKG-2404009,DRKG,0,Generalised,Homo sapiens
4,NUP37,Gene_BiologicalProcess,GO:0071705,Gene,GpBP,BiologicalProcess,TARKG,nucleoporin 37,nitrogen compound transport,NCBI_ID,Quick_GO,Hetionet-149653,Hetionet,0,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1045583,SLC35B4,Gene_BiologicalProcess,GO:1990569,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,TARKG,solute carrier family 35 member B4,UDP-N-acetylglucosamine transmembrane transport,NCBI_ID,Quick_GO,DRKG-2332823,DRKG,0,Generalised,Homo sapiens
1045584,PCYOX1,Gene_BiologicalProcess,GO:0030328,Gene,GpBP,BiologicalProcess,TARKG,prenylcysteine oxidase 1,prenylcysteine catabolic process,NCBI_ID,Quick_GO,Hetionet-295606,Hetionet,0,Generalised,Homo sapiens
1045585,PCYOX1,Gene_BiologicalProcess,GO:0030328,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,TARKG,prenylcysteine oxidase 1,prenylcysteine catabolic process,NCBI_ID,Quick_GO,DRKG-2166808,DRKG,0,Generalised,Homo sapiens
1045586,PCYOX1L,Gene_BiologicalProcess,GO:0030328,Gene,GpBP,BiologicalProcess,TARKG,prenylcysteine oxidase 1 like,prenylcysteine catabolic process,NCBI_ID,Quick_GO,Hetionet-177161,Hetionet,0,Generalised,Homo sapiens


## 3 · Consolidate

In [26]:
source_dfs = [
    AgeAnno, AgingAtlas,
    DRKG, genage, GPKG, Hetionet, Monarch, TARKG
]
aligned = []
for df in source_dfs:
    df = df.copy()
    df = df.loc[:, ~df.columns.duplicated()]
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])
final_df = pd.concat(aligned, ignore_index=True)
final_df = final_df[final_df['head'].astype(str).str.upper() != 'NAN']

In [27]:
final_df[final_df['head'] == 'NAT2']

,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
202794,NAT2,Gene_BiologicalProcess,GO:0006805,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,DRKG,Generalised,NCBI_ID,Quick_GO,10,xenobiotic metabolic process,Homo sapiens
330138,NAT2,Gene_BiologicalProcess,GO:0009410,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,DRKG,Generalised,NCBI_ID,Quick_GO,10,response to xenobiotic stimulus,Homo sapiens
385334,NAT2,Gene_BiologicalProcess,GO:0071466,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,DRKG,Generalised,NCBI_ID,Quick_GO,10,cellular response to xenobiotic stimulus,Homo sapiens
855944,NAT2,Gene_BiologicalProcess,GO:0006805,Gene,Gene–participates–BiologicalProcess,BiologicalProcess,Hetionet,Generalised,NCBI_ID,Quick_GO,N-acetyltransferase 2,xenobiotic metabolic process,Homo sapiens
983288,NAT2,Gene_BiologicalProcess,GO:0009410,Gene,Gene–participates–BiologicalProcess,BiologicalProcess,Hetionet,Generalised,NCBI_ID,Quick_GO,N-acetyltransferase 2,response to xenobiotic stimulus,Homo sapiens
1038484,NAT2,Gene_BiologicalProcess,GO:0071466,Gene,Gene–participates–BiologicalProcess,BiologicalProcess,Hetionet,Generalised,NCBI_ID,Quick_GO,N-acetyltransferase 2,cellular response to xenobiotic stimulus,Homo sapiens
1257533,NAT2,Gene_BiologicalProcess,GO:0006805,Gene,actively_involved_in,BiologicalProcess,Monarch_KG,Generalised,NCBI_ID,Quick_GO,solute carrier family 38 member 1,xenobiotic metabolic process,Homo sapiens
1257534,NAT2,Gene_BiologicalProcess,GO:0006805,Gene,actively_involved_in,BiologicalProcess,Monarch_KG,Generalised,NCBI_ID,Quick_GO,solute carrier family 38 member 1,xenobiotic metabolic process,Homo sapiens
1747205,NAT2,Gene_BiologicalProcess,GO:0009410,Gene,GpBP,BiologicalProcess,TARKG,Generalised,NCBI_ID,Quick_GO,N-acetyltransferase 2,response to xenobiotic stimulus,Homo sapiens
1747206,NAT2,Gene_BiologicalProcess,GO:0009410,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,TARKG,Generalised,NCBI_ID,Quick_GO,N-acetyltransferase 2,response to xenobiotic stimulus,Homo sapiens


In [28]:
final_df['head_detail_name'] = final_df['head'].map(NCBI_sym2desc_dict)
final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,EIF1,Gene_BiologicalProcess,GO:1111113,Gene,None,BiologicalProcess,AgeAnnoMO,Aging,NCBI_ID,Quick_GO,eukaryotic translation initiation factor 1,Epigenetic Alterations,Homo sapiens
1,SELPLG,Gene_BiologicalProcess,GO:1111113,Gene,None,BiologicalProcess,AgeAnnoMO,Aging,NCBI_ID,Quick_GO,selectin P ligand,Epigenetic Alterations,Homo sapiens
2,LINC01592,Gene_BiologicalProcess,GO:1111113,Gene,None,BiologicalProcess,AgeAnnoMO,Aging,NCBI_ID,Quick_GO,long intergenic non-protein coding RNA 1592,Epigenetic Alterations,Homo sapiens
3,PDE3A,Gene_BiologicalProcess,GO:1111113,Gene,None,BiologicalProcess,AgeAnnoMO,Aging,NCBI_ID,Quick_GO,phosphodiesterase 3A,Epigenetic Alterations,Homo sapiens
4,OSBPL3,Gene_BiologicalProcess,GO:1111113,Gene,None,BiologicalProcess,AgeAnnoMO,Aging,NCBI_ID,Quick_GO,oxysterol binding protein like 3,Epigenetic Alterations,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2426536,SLC35B4,Gene_BiologicalProcess,GO:1990569,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,TARKG,Generalised,NCBI_ID,Quick_GO,solute carrier family 35 member B4,UDP-N-acetylglucosamine transmembrane transport,Homo sapiens
2426537,PCYOX1,Gene_BiologicalProcess,GO:0030328,Gene,GpBP,BiologicalProcess,TARKG,Generalised,NCBI_ID,Quick_GO,prenylcysteine oxidase 1,prenylcysteine catabolic process,Homo sapiens
2426538,PCYOX1,Gene_BiologicalProcess,GO:0030328,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,TARKG,Generalised,NCBI_ID,Quick_GO,prenylcysteine oxidase 1,prenylcysteine catabolic process,Homo sapiens
2426539,PCYOX1L,Gene_BiologicalProcess,GO:0030328,Gene,GpBP,BiologicalProcess,TARKG,Generalised,NCBI_ID,Quick_GO,prenylcysteine oxidase 1 like,prenylcysteine catabolic process,Homo sapiens


In [29]:
final_df[final_df['head'] == 'ADA']

,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
2115,ADA,Gene_BiologicalProcess,GO:0009394,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,DRKG,Generalised,NCBI_ID,Quick_GO,adenosine deaminase,2'-deoxyribonucleotide metabolic process,Homo sapiens
5295,ADA,Gene_BiologicalProcess,GO:0009167,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,DRKG,Generalised,NCBI_ID,Quick_GO,adenosine deaminase,purine ribonucleoside monophosphate metabolic ...,Homo sapiens
8592,ADA,Gene_BiologicalProcess,GO:0042113,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,DRKG,Generalised,NCBI_ID,Quick_GO,adenosine deaminase,B cell activation,Homo sapiens
11480,ADA,Gene_BiologicalProcess,GO:0019439,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,DRKG,Generalised,NCBI_ID,Quick_GO,adenosine deaminase,obsolete aromatic compound catabolic process,Homo sapiens
11750,ADA,Gene_BiologicalProcess,GO:0033628,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,DRKG,Generalised,NCBI_ID,Quick_GO,adenosine deaminase,regulation of cell adhesion mediated by integrin,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2417960,ADA,Gene_BiologicalProcess,GO:0046061,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,TARKG,Generalised,NCBI_ID,Quick_GO,adenosine deaminase,dATP catabolic process,Homo sapiens
2417965,ADA,Gene_BiologicalProcess,GO:0042323,Gene,GpBP,BiologicalProcess,TARKG,Generalised,NCBI_ID,Quick_GO,adenosine deaminase,negative regulation of circadian sleep/wake cy...,Homo sapiens
2417966,ADA,Gene_BiologicalProcess,GO:0042323,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,TARKG,Generalised,NCBI_ID,Quick_GO,adenosine deaminase,negative regulation of circadian sleep/wake cy...,Homo sapiens
2417969,ADA,Gene_BiologicalProcess,GO:0070256,Gene,GpBP,BiologicalProcess,TARKG,Generalised,NCBI_ID,Quick_GO,adenosine deaminase,negative regulation of mucus secretion,Homo sapiens


## 4 · Gene Symbol Normalisation & Pre-dedup NA audit

In [30]:


final_df = final_df.dropna(subset=['head_detail_name', 'tail_detail_name']).reset_index(drop=True)

print(f"Consolidated rows (before deduplication): {len(final_df):,}")
print("\nNaN audit before deduplication:")
print(final_df.isna().sum())

Consolidated rows (before deduplication): 2,426,310

NaN audit before deduplication:
head                   0
relation               0
tail                   0
head_type              0
relation_type       2052
tail_type              0
kg_source              0
kg_type                0
head_id_is             0
tail_id_is             0
head_detail_name       0
tail_detail_name       0
species                0
dtype: int64


## 5 · Deduplication & Post-dedup NA audit

In [31]:
def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']
final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type': 'first',
    'relation_type': 'first',
    'tail_type': 'first',
    'kg_source': merge_sources,
    'kg_type': merge_sources,
    'head_id_is': 'first',
    'tail_id_is': 'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'species': 'first'
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
print("\nNaN audit after deduplication:")
print(final_df_dedup.isna().sum())

Before dedup: 2,426,310  |  After dedup: 620,622

NaN audit after deduplication:
head                   0
relation               0
tail                   0
head_type              0
relation_type       1739
tail_type              0
kg_source              0
kg_type                0
head_id_is             0
tail_id_is             0
head_detail_name       0
tail_detail_name       0
species                0
dtype: int64


In [32]:
final_df_dedup

,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,A1BG,Gene_BiologicalProcess,GO:0002764,Gene,actively_involved_in,BiologicalProcess,Monarch_KG,Generalised,NCBI_ID,Quick_GO,alpha-1-B glycoprotein,immune response-regulating signaling pathway,Homo sapiens
1,A1CF,Gene_BiologicalProcess,GO:0006396,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,DRKG::Hetionet::TARKG,Generalised,NCBI_ID,Quick_GO,APOBEC1 complementation factor,RNA processing,Homo sapiens
2,A1CF,Gene_BiologicalProcess,GO:0006397,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,DRKG::Hetionet::Monarch_KG::TARKG,Generalised,NCBI_ID,Quick_GO,APOBEC1 complementation factor,mRNA processing,Homo sapiens
3,A1CF,Gene_BiologicalProcess,GO:0007566,Gene,actively_involved_in,BiologicalProcess,Monarch_KG,Generalised,NCBI_ID,Quick_GO,APOBEC1 complementation factor,embryo implantation,Homo sapiens
4,A1CF,Gene_BiologicalProcess,GO:0009451,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,DRKG::Hetionet::TARKG,Generalised,NCBI_ID,Quick_GO,APOBEC1 complementation factor,RNA modification,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
620617,ZZZ3,Gene_BiologicalProcess,GO:0045995,Gene,associated,BiologicalProcess,GPKG::Monarch_KG,Generalised,NCBI_ID,Quick_GO,zinc finger ZZ-type containing 3,regulation of embryonic development,Homo sapiens
620618,ZZZ3,Gene_BiologicalProcess,GO:0051276,Gene,Hetionet::GpBP::Gene:Biological Process,BiologicalProcess,DRKG::Hetionet::TARKG,Generalised,NCBI_ID,Quick_GO,zinc finger ZZ-type containing 3,chromosome organization,Homo sapiens
620619,ZZZ3,Gene_BiologicalProcess,GO:0051302,Gene,associated,BiologicalProcess,GPKG::Monarch_KG,Generalised,NCBI_ID,Quick_GO,zinc finger ZZ-type containing 3,regulation of cell division,Homo sapiens
620620,ZZZ3,Gene_BiologicalProcess,GO:0051726,Gene,associated,BiologicalProcess,GPKG::Monarch_KG,Generalised,NCBI_ID,Quick_GO,zinc finger ZZ-type containing 3,regulation of cell cycle,Homo sapiens


In [33]:
set(final_df_dedup['kg_source']),set(final_df_dedup['kg_type'])

({'AgeAnnoMO',
  'AgingAtlas',
  'AgingAtlas::DRKG::GPKG::GenAge::Hetionet',
  'AgingAtlas::DRKG::GenAge::Hetionet',
  'AgingAtlas::DRKG::Hetionet',
  'AgingAtlas::GPKG',
  'AgingAtlas::GPKG::GenAge',
  'AgingAtlas::GenAge',
  'DRKG::GPKG::Hetionet',
  'DRKG::GPKG::Hetionet::Monarch_KG',
  'DRKG::GPKG::Hetionet::Monarch_KG::TARKG',
  'DRKG::GPKG::Hetionet::TARKG',
  'DRKG::Hetionet',
  'DRKG::Hetionet::Monarch_KG',
  'DRKG::Hetionet::Monarch_KG::TARKG',
  'DRKG::Hetionet::TARKG',
  'GPKG',
  'GPKG::Monarch_KG',
  'Monarch_KG'},
 {'Aging', 'Aging::Generalised', 'Generalised'})

## 6 · Save Output

In [34]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 620,622 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/GENE_BIOLOGICALPROCESS/ALL_GENE_BIOLOGICALPROCESS.csv
